In [ ]:
import argparse
import statistics
import time
from pathlib import Path

import numpy as np


def countMatches_original(inferred, ground_truth):
    """Original SEE counting algorithm, preserved for comparison."""
    assert inferred.shape == ground_truth.shape
    m = set()
    n = set()
    setcounts = {}
    for r in range(inferred.shape[0]):
        for c in range(inferred.shape[1]):
            i_key = inferred[r, c]
            m.add(i_key)
            g_key = ground_truth[r, c]
            n.add(g_key)
            if i_key in setcounts:
                if g_key in setcounts[i_key]:
                    setcounts[i_key][g_key] += 1
                else:
                    setcounts[i_key][g_key] = 1
            else:
                setcounts[i_key] = {g_key: 1}
    return setcounts, len(m), len(n)


def countMatches_numpy(inferred, ground_truth):
    """Vectorized overlap counting, preserving raster-order insertion for ties."""
    assert inferred.shape == ground_truth.shape
    pred = np.asarray(inferred).ravel()
    gt = np.asarray(ground_truth).ravel()
    pred_labels, pred_idx = np.unique(pred, return_inverse=True)
    gt_labels, gt_idx = np.unique(gt, return_inverse=True)
    m, n = len(pred_labels), len(gt_labels)
    pairs = pred_idx * n + gt_idx
    unique_pairs, first, counts = np.unique(
        pairs, return_index=True, return_counts=True
    )
    setcounts = {}
    for k in np.argsort(first):
        i, j = divmod(int(unique_pairs[k]), n)
        pl = pred_labels[i].item()
        gl = gt_labels[j].item()
        setcounts.setdefault(pl, {})[gl] = int(counts[k])
    return setcounts, m, n


def locate_gt():
    for base in [Path.cwd(), *Path.cwd().parents]:
        p = base / 'Image_data/Examples/AA_Chameleon_GT.png'
        if p.exists():
            return p
    return None


def load_gt(height, width):
    p = locate_gt()
    if p is not None:
        try:
            import imageio.v2 as imageio
            gt = imageio.imread(p)
            if gt.ndim == 3:
                pixels = gt.reshape(-1, gt.shape[-1])
                _, ids = np.unique(pixels, axis=0, return_inverse=True)
                gt = ids.reshape(gt.shape[:2])
            print(f'Ground truth: {p} ({gt.shape}, {len(np.unique(gt))} labels)')
            return gt
        except ImportError:
            print('imageio not installed; using synthetic ground truth instead.')
    print('Using synthetic ground truth (no project PNG found).')
    yy, xx = np.indices((height, width))
    return ((yy - height / 2)**2 / (height / 3)**2 +
            (xx - width / 2)**2 / (width / 3)**2 < 1).astype(np.uint8)


def timed(fn, pred, gt, repeats):
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        result = fn(pred, gt)
        times.append(time.perf_counter() - start)
    return result, statistics.median(times), times


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--repeats', type=int, default=3)
    parser.add_argument('--height', type=int, default=720)
    parser.add_argument('--width', type=int, default=1280)
    parser.add_argument('--segments', type=int, default=38,
                        help='Approximate number of inferred labels')
    args = parser.parse_args(args=[])
    if args.repeats < 1 or args.segments < 1:
        parser.error('--repeats and --segments must be positive')

    gt = load_gt(args.height, args.width)
    h, w = gt.shape
    # Deterministic superpixel-like grid, with a similar label count to SLIC.
    cols = max(1, round(np.sqrt(args.segments * w / h)))
    rows = max(1, round(args.segments / cols))
    yy, xx = np.indices((h, w))
    inferred = (yy * rows // h) * cols + (xx * cols // w)
    print(f'Predicted labels: {len(np.unique(inferred))}; pixels: {h*w:,}')

    old_result, old_median, old_times = timed(
        countMatches_original, inferred, gt, args.repeats
    )
    np_result, np_median, np_times = timed(
        countMatches_numpy, inferred, gt, args.repeats
    )
    matches = old_result == np_result
    print(f'Original median: {old_median:.4f} seconds ({old_times})')
    print(f'NumPy median:    {np_median:.4f} seconds ({np_times})')
    print(f'Speedup:         {old_median / np_median:.2f}x')
    print(f'Exact counts match: {matches}')
    if not matches:
        raise SystemExit('FAIL: implementations disagree; do not replace SEE function')
    print('PASS: NumPy implementation matches original overlap counts.')


main()

Ground truth: /Users/jamesmcalvey/see-segment/Image_data/Examples/AA_Chameleon_GT.png ((720, 1280), 2 labels)
Predicted labels: 40; pixels: 921,600
Original median: 0.3197 seconds ([0.32005204097367823, 0.31633879197761416, 0.3197499170200899])
NumPy median:    0.0922 seconds ([0.09219687501899898, 0.10773491708096117, 0.09071445895824581])
Speedup:         3.47x
Exact counts match: True
PASS: NumPy implementation matches original overlap counts.
